In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "sanchez2016differences")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "sanchez2016differences_experiment1.csv")
complete_path_2 = os.path.join(original_data_pathway, "data all species per individualexp2.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)


data_frames = [df1,df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"species": "species_original",
                    "sex":"sex_original"}, inplace=True) 
    x['participant'] = x['participant'].str.rstrip() ##remove spaces
    x['study_id']="sanchez2016differences"
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')


In [4]:
experiment_rename = [["one",'1'],
                        ["two",'2']]
for x,y in experiment_rename:
    fulldf['experiment'].replace(x, y, inplace=True)

fulldf.columns = fulldf.columns.str.replace('&', '_and_', regex=True)

fulldf['comment'].replace(' ', '_', inplace=True, regex=True)
# fulldf.columns

In [5]:


fulldf=fulldf[['study_id', 'experiment', 'year', 'participant', 'sex','species',
        'pellet_vs_apple', 'pellet_vs_carrot',
       'apple_vs_carrot', '2pellets_vs_1pellet', '2apples_vs_1apple',
       '2carrots_vs_1carrot', 'pellet_and_apple_vs_pellet',
       'pellet_and_carrot_vs_pellet', 'apple_and_carrot_vs_apple', 
       'pellet_and_banana_vs_pellet', 'pellet_and_grape_vs_pellet',
       'banana_and_pellet_vs_banana', 'banana_and_grape_vs_banana',
       'grape_and_pellet_vs_grape', 'grape_and_banana_vs_grape',
       'pellet_vs_banana', 'pellet_vs_grape', 'banana_vs_grape', 'comment']]

for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'sanchez2016differences_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'sanchez2016differences_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
